# RTX 5090 — Murmur Mamba3 MIMO ~350M: обучение с нуля

Этот ноутбук создаёт всё с нуля: смешанный потоковый корпус 7.5B токенов, BPE-токенизатор, бинарные шарды и чекпойнты. Архитектура: GQA Prelude (2) → shared Mamba3 MIMO Core (3) × depth 1–4 → GQA Coda (2). Нужны RTX 5090, 32 GB VRAM, CUDA 13.0 / PyTorch 2.9.0+cu130, минимум 200 GB диска и включённый Internet.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO=Path('/workspace/murmur-science')
BRANCH='codex/rtx5090-mimo-smoke'
if not REPO.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'https://github.com/orkrs/murmur-science.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin',BRANCH],check=True)
os.chdir(REPO)
sys.path.insert(0,str(REPO/'src'))
print('Repository:', REPO, 'branch:', BRANCH)

In [ ]:
# Hardware gate. Stop here if the container does not expose the intended Blackwell stack.
import torch
if not torch.cuda.is_available(): raise RuntimeError('CUDA GPU is required')
if 'RTX 5090' not in torch.cuda.get_device_name(0): raise RuntimeError(f'Expected RTX 5090, got {torch.cuda.get_device_name(0)}')
if torch.cuda.get_device_capability(0)!=(12,0): raise RuntimeError(f'Expected sm_120, got {torch.cuda.get_device_capability(0)}')
if torch.__version__ != '2.9.0+cu130': raise RuntimeError(f'Use the PyTorch 2.9.0+cu130 CUDA 13.0 Vast image, got {torch.__version__}')
if not str(torch.version.cuda).startswith('13.0'): raise RuntimeError(f'Expected CUDA 13.0, got {torch.version.cuda}')
print({'gpu':torch.cuda.get_device_name(0),'torch':torch.__version__,'cuda':torch.version.cuda,'vram_gb':round(torch.cuda.get_device_properties(0).total_memory/2**30,1)})

In [ ]:
%pip install -q --upgrade datasets sentencepiece pyarrow pandas einops ninja safetensors
%pip install -q --upgrade 'tilelang==0.1.9' 'apache-tvm-ffi<=0.1.12' 'quack-kernels>=0.3.4' 'triton>=3.5.0' 'nvidia-cutlass-dsl'
os.environ['MAMBA_FORCE_BUILD']='TRUE'
%pip install -q --no-cache-dir --no-deps --force-reinstall --no-build-isolation 'mamba-ssm==2.3.2.post1'
import importlib.metadata
assert importlib.metadata.version('tilelang') == '0.1.9'
assert importlib.metadata.version('mamba-ssm') == '2.3.2.post1'
print('Pinned Blackwell MIMO stack is ready')

In [ ]:
# Mandatory: exact production recurrent-core forward + backward before downloading 7.5B tokens.
from mamba_ssm.modules.mamba3 import Mamba3
torch.cuda.reset_peak_memory_stats()
core=Mamba3(d_model=1792,d_state=128,headdim=64,is_mimo=True,mimo_rank=2,chunk_size=32,dtype=torch.bfloat16).cuda().train()
x=torch.randn(1,1024,1792,device='cuda',dtype=torch.bfloat16,requires_grad=True)
loss=core(x).float().square().mean(); loss.backward(); torch.cuda.synchronize()
assert torch.isfinite(loss)
print({'mimo_350m_core_gate':'passed','peak_vram_gb':round(torch.cuda.max_memory_allocated()/2**30,2)})
del core, x, loss; torch.cuda.empty_cache()

In [ ]:
# Build the complete, weighted EN/RU/code/math/reasoning dataset directly from Hugging Face.
CORPUS=Path('artifacts/rtx5090_mixed_350m_corpus')
if not (CORPUS/'provenance.json').exists():
    subprocess.run([sys.executable,'scripts/build_hf_mix.py','--profile','mixed_350m','--output',str(CORPUS),'--max-tokens','7500000000'],check=True)
else:
    print('Corpus already exists — keeping it for resumability')
print((CORPUS/'provenance.json').read_text(encoding='utf-8')[:4000])

In [ ]:
TOKENIZER=Path('artifacts/rtx5090_mixed_350m_tokenizer.model')
if not TOKENIZER.exists():
    subprocess.run([sys.executable,'scripts/train_tokenizer.py','--corpus','artifacts/rtx5090_mixed_350m_corpus/corpus.txt','--output',str(TOKENIZER),'--vocab-size','48000'],check=True)
else:
    print('Tokenizer already exists — keeping it for resumability')
DATA=Path('artifacts/rtx5090_mixed_350m_data')
if not list(DATA.glob('train*.bin')):
    subprocess.run([sys.executable,'scripts/prepare_data.py','--config','configs/rtx5090_mimo_350m.toml','--tokenizer',str(TOKENIZER),'--train-input','artifacts/rtx5090_mixed_350m_corpus/train.jsonl','--val-input','artifacts/rtx5090_mixed_350m_corpus/val.jsonl','--output',str(DATA)],check=True)
else:
    print('Packed data already exists — keeping it for resumability')

In [ ]:
# Count the actual official-Mamba implementation after it is installed.
subprocess.run([sys.executable,'scripts/param_count.py','--config','configs/rtx5090_mimo_350m.toml'],check=True)
from murmur.config import load_run_config
cfg=load_run_config(Path('configs/rtx5090_mimo_350m.toml'))
assert cfg.model.mixer=='mamba3_mimo' and cfg.model.n_core==3 and cfg.train.max_tokens==7_500_000_000
print('Production configuration verified')

In [ ]:
# Start from scratch or resume the last complete atomic checkpoint automatically.
RUN=Path('artifacts/runs/rtx5090_mimo_350m')
LAST=RUN/'checkpoints'/'last'
command=[sys.executable,'scripts/train.py','--config','configs/rtx5090_mimo_350m.toml','--run-dir',str(RUN),'--device','cuda']
if (LAST/'COMPLETED').exists(): command += ['--resume',str(LAST)]
print('Running:', ' '.join(command))
subprocess.run(command,check=True)
assert (LAST/'COMPLETED').exists(), 'training did not publish a complete checkpoint'
print('Training finished; final checkpoint:', LAST)